In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import re

In [ ]:
df_new = pd.read_csv('~/data/combined_season_results_cleaned.csv')
stats_df = pd.read_csv('~/data/nfl_sentiment_2025_cleaned.csv')

df_new["created_at"] = pd.to_datetime(df_new["created_at"])

In [ ]:
# Make sure kickoff times are timestamps
stats_df = stats_df.copy()
stats_df["game_start_time"] = pd.to_datetime(stats_df["game_start_time"], utc=True)

# Build a unique game table: team + week -> kickoff
game_lookup = (
    stats_df[["team_abbreviation", "game_week", "game_start_time"]]
    .drop_duplicates(subset=["team_abbreviation", "game_week"])
    .rename(columns={
        "team_abbreviation": "team_abbr",
        "game_week": "week"
    })
)

team_name_to_abbr = {
    "Eagles": "PHI",
    "Patriots": "NE",
    "Cowboys": "DAL",
    "Bears": "CHI",
    "Bills": "BUF",
    "Seahawks": "SEA",
    "Bengals": "CIN",
    "Chiefs": "KC",
    "Colts": "IND",
    "Buccaneers": "TB"
}

df_new = df_new.copy()

df_new["week"] = (
    df_new["game_id"]
    .str.extract(r"W(\d+)")
    .astype(int)
)

df_new["created_at"] = pd.to_datetime(df_new["created_at"], utc=True)

df_new["team_abbr"] = df_new["team"].map(team_name_to_abbr)

df_new = df_new.merge(
    game_lookup,
    how="left",
    left_on=["team_abbr", "week"],
    right_on=["team_abbr", "week"]
)

df_new["minutes_since_kickoff"] = (
    (df_new["created_at"] - df_new["game_start_time"])
    .dt.total_seconds() / 60
)

In [ ]:
df_new[[
    "team", "player", "game_id",
    "created_at", "game_start_time",
    "minutes_since_kickoff"
]].head(10)

df_new['hours_since_kickoff'] = df_new['minutes_since_kickoff'] / 60

#  make a histogram of the hours since kickoff
import matplotlib.pyplot as plt
plt.hist(df_new['hours_since_kickoff'], bins=24, edgecolor='black')
plt.xlabel('Hours Since Kickoff')
plt.ylabel('Number of Tweets')
plt.title('Histogram of Hours Since Kickoff for Tweets')
plt.show()

# make a box plot of the hours since kickoff
plt.boxplot(df_new['hours_since_kickoff'])
plt.xlabel('Hours Since Kickoff')
plt.ylabel('Number of Tweets')
plt.title('Box Plot of Hours Since Kickoff for Tweets')
plt.show()

In [ ]:
# pre-game tweets only
pre_df = df_new[df_new["is_post_game"] == False]

# average pre-game sentiment per player-game
pre_sent = (
    pre_df
    .groupby(["player", "game_id"], as_index=False)["sentiment_scores"]
    .mean()
    .rename(columns={"sentiment_scores": "pre_game_sentiment"})
)

# merge with fantasy performance
plot_df = pre_sent.merge(
    stats_df[["player_name", "game_id", "fantasy_points_ppr"]],
    left_on=["player", "game_id"],
    right_on=["player_name", "game_id"],
    how="inner"
)

plt.figure(figsize=(7, 5))
plt.scatter(
    plot_df["pre_game_sentiment"],
    plot_df["fantasy_points_ppr"],
    alpha=0.6
)

plt.xlabel("Pre-game sentiment")
plt.ylabel("Fantasy points (PPR)")
plt.title("Pre-game sentiment vs next-game fantasy performance")
plt.tight_layout()
plt.show()